In [185]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [186]:
RAW_DIR = "../data/wisdm-dataset/raw"
SUBJECTS = range(1600, 1651)  # 51 subjects
SENSORS = [("phone", "accel"), ("phone", "gyro"), ("watch", "accel"), ("watch", "gyro")]

In [187]:
def get_sensor_data(filepath):
    df = pd.read_csv(
        filepath,
        header=None,
        names=["subject_id", "activity", "timeStamp", "x", "y", "z"],
    )

    df.iloc[:, -1] = df.iloc[:, -1].str.rstrip(";")

    df = df.astype({"z": float})

    df["timeStamp"] = pd.to_datetime(df["timeStamp"], unit="ns")
    df.drop(columns="subject_id", inplace=True)

    return df

In [188]:
def get_subject_data(subject_id):
    dfs = {}
    for device, sensor in SENSORS:

        df = get_sensor_data(
            f"{RAW_DIR}/{device}/{sensor}/data_{subject_id}_{sensor}_{device}.txt"
        )

        suffix = f"{device}_{sensor}"

        df = df.rename(
            columns={
                "x": f"x_{suffix}",
                "y": f"y_{suffix}",
                "z": f"z_{suffix}",
            }
        )

        dfs[suffix] = df

    sensors_list = list(dfs.keys())
    merged = dfs[sensors_list[0]].copy()

    merged.sort_values(by="timeStamp", inplace=True)

    for sensor in sensors_list[1:]:
        other = dfs[sensor].copy()
        other.sort_values(by="timeStamp", inplace=True)

        merged = pd.merge_asof(
            merged,
            other,
            on="timeStamp",
            by="activity",
            direction="nearest",
            tolerance=pd.Timedelta("1s")
        )

    return merged.sort_values(by=["activity", "timeStamp"])

In [197]:
df = get_subject_data(1600)

df


,activity,timeStamp,x_phone_accel,y_phone_accel,z_phone_accel,x_phone_gyro,y_phone_gyro,z_phone_gyro,x_watch_accel,y_watch_accel,z_watch_accel,x_watch_gyro,y_watch_gyro,z_watch_gyro
39300,A,1970-01-03 22:03:27.666810782,-0.364761,8.793503,1.055084,-0.853210,0.297226,0.890182,NaN,NaN,NaN,NaN,NaN,NaN
39301,A,1970-01-03 22:03:27.717164786,-0.879730,9.768784,1.016998,-0.853210,0.297226,0.890182,NaN,NaN,NaN,NaN,NaN,NaN
39302,A,1970-01-03 22:03:27.767518790,2.001495,11.109070,2.619156,-0.853210,0.297226,0.890182,NaN,NaN,NaN,NaN,NaN,NaN
39303,A,1970-01-03 22:03:27.817872794,0.450623,12.651642,0.184555,-0.853210,0.297226,0.890182,NaN,NaN,NaN,NaN,NaN,NaN
39304,A,1970-01-03 22:03:27.868226798,-2.164352,13.928436,-4.422485,-0.853210,0.297226,0.890182,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17857,S,1970-01-03 19:55:05.660942200,-2.372223,9.242722,-1.588287,0.376999,-0.444031,-0.142746,NaN,NaN,NaN,NaN,NaN,NaN
17858,S,1970-01-03 19:55:05.711296204,-2.046921,10.032288,-1.229935,0.489685,-0.599976,-0.248138,NaN,NaN,NaN,NaN,NaN,NaN
17859,S,1970-01-03 19:55:05.761650208,-1.393539,9.883896,-0.479248,0.346771,-0.459473,-0.254272,NaN,NaN,NaN,NaN,NaN,NaN
17860,S,1970-01-03 19:55:05.812004212,-1.230454,9.315079,-0.155701,0.279709,-0.136780,-0.331009,NaN,NaN,NaN,NaN,NaN,NaN
